# Task 3: Gated DeltaNet - Fashion-MNIST 训练

**运行前请确认**：菜单 `运行时 → 更改运行时类型 → T4 GPU`

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 模型定义

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ═══ Zero-Centered RMSNorm ═══

class ZeroCenteredRMSNorm(nn.Module):
    """增益参数 gamma 初始化为 0，初始时等价于纯 RMSNorm"""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * (1.0 + self.gamma)


# ═══ Gated DeltaNet 核心层 (Recurrent 模式) ═══

class GatedDeltaNet(nn.Module):
    """
    核心更新规则：
        S_t = S_{t-1} @ (alpha_t (I - beta_t k_t k_t^T)) + beta_t v_t k_t^T
        o_t = S_t @ q_t
    """

    def __init__(self, hidden_dim: int, num_heads: int, head_dim: int = None, conv_size: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = head_dim or hidden_dim // num_heads
        self.inner_dim = num_heads * self.head_dim
        self.conv_size = conv_size

        # 线性投影
        self.q_proj = nn.Linear(hidden_dim, self.inner_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, self.inner_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, self.inner_dim, bias=False)
        self.alpha_proj = nn.Linear(hidden_dim, num_heads, bias=True)   # 衰减门
        self.beta_proj  = nn.Linear(hidden_dim, num_heads, bias=True)   # 输入门
        self.gate_proj = nn.Linear(hidden_dim, self.inner_dim, bias=False)  # 输出门
        self.out_proj  = nn.Linear(self.inner_dim, hidden_dim, bias=False)

        # 深度可分离因果卷积
        self.q_conv = nn.Conv1d(self.inner_dim, self.inner_dim, kernel_size=conv_size, padding=0, groups=self.inner_dim, bias=True)
        self.k_conv = nn.Conv1d(self.inner_dim, self.inner_dim, kernel_size=conv_size, padding=0, groups=self.inner_dim, bias=True)
        self.v_conv = nn.Conv1d(self.inner_dim, self.inner_dim, kernel_size=conv_size, padding=0, groups=self.inner_dim, bias=True)

        # 输出归一化
        self.norm = ZeroCenteredRMSNorm(self.inner_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.constant_(self.alpha_proj.bias, 2.0)  # sigmoid ≈ 0.88，默认记忆历史
        nn.init.constant_(self.beta_proj.bias, 0.0)

    def _causal_conv(self, x: torch.Tensor, conv: nn.Conv1d) -> torch.Tensor:
        x = x.transpose(1, 2)                           # (B, C, T)
        x = F.pad(x, (self.conv_size - 1, 0))           # 左填充保证因果性
        x = conv(x)
        return x.transpose(1, 2)                         # (B, T, C)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, d = self.num_heads, self.head_dim

        # 1. 投影 + 短卷积 + 激活
        q = F.silu(self._causal_conv(self.q_proj(x), self.q_conv))
        k = F.silu(self._causal_conv(self.k_proj(x), self.k_conv))
        v = F.silu(self._causal_conv(self.v_proj(x), self.v_conv))

        # 2. reshape + L2归一化
        q = F.normalize(q.view(B, T, H, d), p=2, dim=-1)
        k = F.normalize(k.view(B, T, H, d), p=2, dim=-1)
        v = v.view(B, T, H, d)

        # 3. 标量门控
        alpha = torch.sigmoid(self.alpha_proj(x))   # (B, T, H)
        beta  = torch.sigmoid(self.beta_proj(x))    # (B, T, H)

        # 4. 输出门
        gate = F.silu(self.gate_proj(x))             # (B, T, inner)

        # 5. 循环计算 Delta Rule
        S = x.new_zeros(B, H, d, d)
        I_d = torch.eye(d, device=x.device, dtype=x.dtype)

        outputs = []
        for t in range(T):
            q_t = q[:, t]                                   # (B, H, d)
            k_t = k[:, t]
            v_t = v[:, t]
            a_t = alpha[:, t, :, None, None]                # (B, H, 1, 1)
            b_t = beta[:, t, :, None, None]

            kk_T = k_t.unsqueeze(-1) * k_t.unsqueeze(-2)   # (B, H, d, d)
            vk_T = v_t.unsqueeze(-1) * k_t.unsqueeze(-2)

            decay = a_t * (I_d - b_t * kk_T)
            S = torch.matmul(S, decay) + b_t * vk_T

            o_t = torch.einsum("bhde,bhe->bhd", S, q_t)
            outputs.append(o_t)

        # 6. 组装、归一化、门控、投影
        output = torch.stack(outputs, dim=1)               # (B, T, H, d)
        output = output.reshape(B, T, self.inner_dim)
        output = self.norm(output) * gate
        output = self.out_proj(output)

        return output


# ═══ MLP ═══

class MLP(nn.Module):
    def __init__(self, dim: int, expansion: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * expansion)
        self.fc2 = nn.Linear(dim * expansion, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))


# ═══ GDN Block (pre-norm residual) ═══

class GDNBlock(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int, mlp_expansion: int = 2):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.gdn   = GatedDeltaNet(hidden_dim, num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.mlp   = MLP(hidden_dim, mlp_expansion)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.gdn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# ═══ Patch Embedding ═══

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size * patch_size * in_channels
        self.proj = nn.Linear(patch_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.contiguous().view(B, -1, C * p * p)
        return self.proj(x)


# ═══ 完整分类器 ═══

class GDNClassifier(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, hidden_dim=64,
                 num_heads=4, num_layers=3, mlp_expansion=2, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, hidden_dim)
        num_patches = self.patch_embed.num_patches

        # 可学习位置编码
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, hidden_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList(
            [GDNBlock(hidden_dim, num_heads, mlp_expansion) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)


print('模型定义完成')

## 2. 数据加载 & 训练

In [ ]:
import os
import time
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ═══ 配置 ═══
CONFIG = {
    "img_size": 28, "patch_size": 4, "hidden_dim": 64,
    "num_heads": 4, "num_layers": 3, "mlp_expansion": 2,
    "num_classes": 10,
    "epochs": 10, "batch_size": 128, "lr": 1e-3, "weight_decay": 1e-4,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ═══ 数据 ═══
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_set = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set)} | Test: {len(test_set)}")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

# ═══ 模型 ═══
torch.manual_seed(42)
model = GDNClassifier(
    img_size=CONFIG["img_size"], patch_size=CONFIG["patch_size"],
    hidden_dim=CONFIG["hidden_dim"], num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"], mlp_expansion=CONFIG["mlp_expansion"],
    num_classes=CONFIG["num_classes"],
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n模型参数量: {n_params:,}")
print(model)

In [ ]:
# ═══ 训练循环 ═══

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

history = {"train_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    epoch_loss = correct = total = 0
    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    train_loss = epoch_loss / total
    train_acc  = correct / total
    test_acc   = evaluate(model, test_loader, device)
    elapsed    = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch:>2d}/{CONFIG['epochs']} | loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_acc {test_acc:.4f} | lr {scheduler.get_last_lr()[0]:.6f} | {elapsed:.1f}s")

print(f"\n\u6700\u7ec8\u6d4b\u8bd5\u51c6\u786e\u7387: {history['test_acc'][-1]:.4f}")

## 3. 可视化 & 保存

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Loss
ax1.plot(epochs_range, history["train_loss"], "o-", markersize=3, label="Train Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, history["train_acc"],  "o-", markersize=3, label="Train Acc")
ax2.plot(epochs_range, history["test_acc"],   "s-", markersize=3, label="Test Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Classification Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("训练曲线已保存为 training_curves.png")

In [ ]:
# 保存模型权重
torch.save(model.state_dict(), "gdn_fashion_mnist.pth")
print("模型已保存为 gdn_fashion_mnist.pth")

print(f"\n{'='*50}")
print(f"模型参数量: {n_params:,}")
print(f"最终测试准确率: {history['test_acc'][-1]:.4f}")
print(f"{'='*50}")

In [ ]:
# 下载文件到本地
from google.colab import files
files.download("training_curves.png")
files.download("gdn_fashion_mnist.pth")